In [9]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import json
from math import hypot
from skimage import io
from skimage.morphology import (skeletonize, remove_small_objects,
                                binary_closing, disk)

In [10]:


# ====================================================
#                PARÁMETROS Y CONSTANTES
# ====================================================
NEIGHBORS_8 = [
    (-1, -1), (-1, 0), (-1, 1),
    ( 0, -1),          ( 0, 1),
    ( 1, -1), ( 1, 0), ( 1, 1)
]

# Rutas de salida (se guardarán en la carpeta "resultados")
IMAGE_OUTPUT_DIR = "./resultadosP2/images/"
JSON_OUTPUT_DIR = "./resultadosP2/json/"

# Asegúrate de que las carpetas existan
os.makedirs(IMAGE_OUTPUT_DIR, exist_ok=True)
os.makedirs(JSON_OUTPUT_DIR, exist_ok=True)

In [11]:

# ====================================================
#                FUNCIONES AUXILIARES
# ====================================================
def get_critical_nodes(G):
    """Devuelve los nodos con grado != 2."""
    return {n for n in G.nodes() if G.degree(n) != 2}

def dist(a, b):
    """Distancia euclidiana entre dos nodos (x1,y1) y (x2,y2)."""
    return hypot(a[0] - b[0], a[1] - b[1])

def collapse_path(path, spacing_nodes=1):
    """
    "Espacia" un camino de nodos intermedios, añadiendo un nodo
    cada vez que se acumule una distancia >= spacing_nodes.
    """
    if not path:
        return []
    collapsed = [path[0]]
    acc_dist = 0
    last = path[0]
    for current in path[1:]:
        d = dist(current, last)
        acc_dist += d
        if acc_dist >= spacing_nodes:
            collapsed.append(current)
            last = current
            acc_dist = 0
    if collapsed[-1] != path[-1]:
        collapsed.append(path[-1])
    return collapsed

def collapse_intermediates(G, spacing_nodes=1, spacing_edges=0):
    """
    Recorre cada camino entre nodos críticos y colapsa los intermedios
    usando 'collapse_path'. 
      - spacing_nodes: controla la densidad de nodos intermedios.
      - spacing_edges: si > 0, filtra (descarta) aristas cuya longitud total
                       es menor que este valor.
    """
    H = nx.Graph()
    critical_nodes = get_critical_nodes(G)
    processed_edges = set()
    
    for start in critical_nodes:
        for neighbor in G.neighbors(start):
            edge_id = tuple(sorted([start, neighbor]))
            if edge_id in processed_edges:
                continue
            # Recorremos el camino hasta llegar a otro nodo crítico
            path = [start, neighbor]
            current = neighbor
            prev = start
            while current not in critical_nodes:
                nbrs = list(G.neighbors(current))
                next_candidates = [n for n in nbrs if n != prev]
                if not next_candidates:
                    break
                next_node = next_candidates[0]
                path.append(next_node)
                prev, current = current, next_node
            
            # Aplica collapse_path usando spacing_nodes
            collapsed = collapse_path(path, spacing_nodes=spacing_nodes)
            
            # Filtrar aristas muy cortas si spacing_edges > 0
            total_dist = sum(dist(path[i], path[i+1]) for i in range(len(path)-1))
            if spacing_edges > 0 and total_dist < spacing_edges:
                processed_edges.add(tuple(sorted([start, path[-1]])))
                continue
            
            for i in range(len(collapsed) - 1):
                H.add_edge(collapsed[i], collapsed[i+1], path=collapsed)
            processed_edges.add(tuple(sorted([start, path[-1]])))
    
    for node in H.nodes():
        if node in G.nodes():
            H.nodes[node]['tipo'] = G.nodes[node].get('tipo', 'desconocido')
    return H


In [12]:
def export_graph_to_json(H, json_path, resumen):
    """
    Exporta la estructura del grafo (nodos, aristas y resumen) a un archivo JSON.
    """
    nodos = []
    for node in H.nodes():
        x, y = node
        tipo = H.nodes[node].get('tipo', 'desconocido')
        nodos.append({
            "coords": [int(x), int(y)],
            "tipo": tipo
        })
    
    aristas = []
    for (n1, n2) in H.edges():
        x1, y1 = n1
        x2, y2 = n2
        aristas.append({
            "start": [int(x1), int(y1)],
            "end":   [int(x2), int(y2)]
        })
    
    data = {
        "resumen": resumen,
        "nodos": nodos,
        "aristas": aristas
    }
    
    with open(json_path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Archivo JSON exportado: {json_path}")

In [13]:




# ====================================================
#                FUNCIONES DE PROCESAMIENTO
# ====================================================
def procesar_imagen(ruta_imagen):
    """Procesa una imagen, genera el grafo, exporta JSON y guarda la imagen final."""
    print(f"Procesando: {ruta_imagen}")
    # Cargar imagen en escala de grises
    img = io.imread(ruta_imagen, as_gray=True)
    
    # --- Preprocesado ---
    binary = img > 0.5
    binary = remove_small_objects(binary, min_size=30)
    binary = binary_closing(binary, disk(2))
    
    # --- Esqueleto ---
    skeleton = skeletonize(binary)
    
    # --- Construir grafo original G (píxel a píxel del esqueleto) ---
    G = nx.Graph()
    rows, cols = skeleton.shape
    for x in range(rows):
        for y in range(cols):
            if skeleton[x, y]:
                G.add_node((x, y))
                for dx, dy in NEIGHBORS_8:
                    nx_ = x + dx
                    ny_ = y + dy
                    if 0 <= nx_ < rows and 0 <= ny_ < cols:
                        if skeleton[nx_, ny_]:
                            G.add_edge((x, y), (nx_, ny_))
    
    # --- Clasificar nodos ---
    for node in G.nodes():
        deg = G.degree(node)
        if deg == 1:
            tipo = 'extremo'
        elif deg == 2:
            tipo = 'intermedio'
        elif deg == 3:
            tipo = 'bifurcacion'
        elif deg >= 4:
            tipo = 'trifurcacion'
        G.nodes[node]['tipo'] = tipo

    # --- Estadísticas ---
    node_types = {n: G.nodes[n]['tipo'] for n in G.nodes()}
    n_extremos = sum(1 for t in node_types.values() if t == 'extremo')
    n_intermedios = sum(1 for t in node_types.values() if t == 'intermedio')
    n_bifurcaciones = sum(1 for t in node_types.values() if t == 'bifurcacion')
    n_trifurcaciones = sum(1 for t in node_types.values() if t == 'trifurcacion')
    
    print("===== RESUMEN DE NODOS =====")
    print(f"Extremos: {n_extremos}")
    print(f"Intermedios: {n_intermedios}")
    print(f"Bifurcaciones: {n_bifurcaciones}")
    print(f"Trifurcaciones: {n_trifurcaciones}")
    print(f"Total de nodos: {G.number_of_nodes()}")
    print(f"Total de aristas: {G.number_of_edges()}")
    
    resumen = {
        "extremos": n_extremos,
        "intermedios": n_intermedios,
        "bifurcaciones": n_bifurcaciones,
        "trifurcaciones": n_trifurcaciones,
        "total_nodos": G.number_of_nodes(),
        "total_aristas": G.number_of_edges()
    }
    
    # --- Colapsar intermedios para simplificar los nodos (H) ---
    spacing_nodes = 250   # Para simplificar la visualización de los nodos
    spacing_edges = 0     # No filtramos aristas cortas
    H = collapse_intermediates(G, spacing_nodes=spacing_nodes, spacing_edges=spacing_edges)
    
    # --- Obtener grafo J para aristas "fieles" al esqueleto ---
    # Con spacing_nodes muy bajo para preservar todos los píxeles
    J = collapse_intermediates(G, spacing_nodes=0.5, spacing_edges=0)
    
    # --- Exportar JSON del grafo simplificado H ---
    nombre_base = os.path.basename(ruta_imagen).replace("_gt.pgm", "")
    json_path = os.path.join(JSON_OUTPUT_DIR, f"{nombre_base}_graph.json")
    export_graph_to_json(H, json_path, resumen)
    
    # --- Visualización y guardado de la imagen final ---
    plt.figure(figsize=(8, 8))
    plt.imshow(binary, cmap='gray', origin='upper')
    
    # Posiciones: (fila, col) -> (col, fila)
    pos_H = {(x, y): (y, x) for (x, y) in H.nodes()}
    pos_J = {(x, y): (y, x) for (x, y) in J.nodes()}
    # Dibujar aristas de J (fieles al esqueleto)
    nx.draw_networkx_edges(J, pos=pos_J, edge_color='yellow', width=3.0)
    
    # Dibujar nodos críticos de H
    critical_nodes = get_critical_nodes(G)
    extremos_nodes = [n for n in critical_nodes if H.nodes[n].get('tipo') == 'extremo']
    bifurcaciones_nodes = [n for n in critical_nodes if H.nodes[n].get('tipo') == 'bifurcacion']
    trifurcaciones_nodes = [n for n in critical_nodes if H.nodes[n].get('tipo') == 'trifurcacion']
    nx.draw_networkx_nodes(H, pos=pos_H, nodelist=extremos_nodes,
                           node_color='limegreen', label='Extremos', node_size=100)
    nx.draw_networkx_nodes(H, pos=pos_H, nodelist=bifurcaciones_nodes,
                           node_color='tomato', label='Bifurcaciones', node_size=100)
    nx.draw_networkx_nodes(H, pos=pos_H, nodelist=trifurcaciones_nodes,
                           node_color='dodgerblue', label='Trifurcaciones', node_size=100)
    intermedios = set(H.nodes()) - critical_nodes
    nx.draw_networkx_nodes(H, pos=pos_H, nodelist=list(intermedios),
                           node_color='gray', label='Intermedios', node_size=50)
    
    plt.xlim(0, cols)
    plt.ylim(rows, 0)
    plt.legend(loc='lower left', bbox_to_anchor=(0, 1.02))
    plt.axis('off')
    
    image_path = os.path.join(IMAGE_OUTPUT_DIR, f"{nombre_base}_graph.png")
    plt.savefig(image_path, dpi=150, bbox_inches='tight')
    print(f"Imagen con grafo guardada: {image_path}")
    plt.close()

def iterar_base_datos(ruta_base):
    """
    Itera sobre todos los archivos que cumplan el patrón *_gt.pgm en la carpeta dada
    y procesa cada imagen.
    """
    patron = os.path.join(ruta_base, "*_gt.pgm")
    archivos = glob.glob(patron)
    print(f"Se encontraron {len(archivos)} archivos que coinciden con el patrón.")
    for archivo in archivos:
        procesar_imagen(archivo)

if __name__ == "__main__":
    # Especifica la carpeta base donde se encuentran las imágenes
    ruta_base = "./database/database/"
    iterar_base_datos(ruta_base)
   


Se encontraron 20 archivos que coinciden con el patrón.
Procesando: ./database/database/10_gt.pgm
===== RESUMEN DE NODOS =====
Extremos: 5
Intermedios: 675
Bifurcaciones: 15
Trifurcaciones: 3
Total de nodos: 698
Total de aristas: 706
Archivo JSON exportado: ./resultadosP2/json/10_graph.json
Imagen con grafo guardada: ./resultadosP2/images/10_graph.png
Procesando: ./database/database/11_gt.pgm
===== RESUMEN DE NODOS =====
Extremos: 6
Intermedios: 872
Bifurcaciones: 32
Trifurcaciones: 3
Total de nodos: 913
Total de aristas: 929
Archivo JSON exportado: ./resultadosP2/json/11_graph.json
Imagen con grafo guardada: ./resultadosP2/images/11_graph.png
Procesando: ./database/database/12_gt.pgm
===== RESUMEN DE NODOS =====
Extremos: 7
Intermedios: 1319
Bifurcaciones: 27
Trifurcaciones: 5
Total de nodos: 1358
Total de aristas: 1373
Archivo JSON exportado: ./resultadosP2/json/12_graph.json
Imagen con grafo guardada: ./resultadosP2/images/12_graph.png
Procesando: ./database/database/13_gt.pgm
=====